In [13]:
import Pkg
for pkg in ["CSV", "DataFrames", "GLMakie", "Colors"]
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

using CSV, DataFrames, GLMakie, Colors

# Load robot log CSV (same folder as this notebook)
df = CSV.read("task2.csv", DataFrame)

# 3D position (linear TCP pose)
x = Float32.(df.actual_TCP_pose_0)
y = Float32.(df.actual_TCP_pose_1)
z = Float32.(df.actual_TCP_pose_2)

# Orientation components used for color (map 3/4/5 to RGB)
a = df.actual_TCP_pose_3
b = df.actual_TCP_pose_4
c = df.actual_TCP_pose_5

# Normalize helper to map any vector to [0, 1]
norm01(v) = begin
    vmin, vmax = minimum(v), maximum(v)
    vmax == vmin ? fill(0.5, length(v)) : (v .- vmin) ./ (vmax - vmin)
end

r = norm01(a)
g = norm01(b)
bl = norm01(c)
point_colors = RGBf.(r, g, bl)
points = Point3f.(x, y, z)

fig = Figure()
ax = Axis3(
    fig[1, 1],
    xlabel = "X [m]",
    ylabel = "Y [m]",
    zlabel = "Z [m]",
    title = "TCP Position (XYZ) colored by TCP Orientation (Rx,Ry,Rz)"
)
scatter!(ax, points; color = point_colors, markersize = 6)

display(GLMakie.Screen(), fig)

GLMakie.Screen(...)

In [10]:
import Pkg
for pkg in ["GLMakie", "Colors", "JSON3"]
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

using Sockets, JSON3, GLMakie, Colors

# Start record.py with:
# python UR5/demo_4_16/record.py -o task2.csv --stream-udp-host 127.0.0.1 --stream-udp-port 9999

listen_host = ip"127.0.0.1"
listen_port = 9999

global LIVE_STOP = isdefined(Main, :LIVE_STOP) ? LIVE_STOP : Ref(false)
global LIVE_TASK = isdefined(Main, :LIVE_TASK) ? LIVE_TASK : Ref{Union{Task, Nothing}}(nothing)
global LIVE_SOCK = isdefined(Main, :LIVE_SOCK) ? LIVE_SOCK : Ref{Union{UDPSocket, Nothing}}(nothing)
global LIVE_HOST = listen_host
global LIVE_PORT = listen_port

global LIVE_SCREEN = isdefined(Main, :LIVE_SCREEN) ? LIVE_SCREEN : Ref{Union{GLMakie.Screen, Nothing}}(nothing)
global LIVE_FIG = isdefined(Main, :LIVE_FIG) ? LIVE_FIG : Ref{Union{Figure, Nothing}}(nothing)
global LIVE_POINTS = isdefined(Main, :LIVE_POINTS) ? LIVE_POINTS : Observable(Point3f[])
global LIVE_COLORS = isdefined(Main, :LIVE_COLORS) ? LIVE_COLORS : Observable(RGBf[])

if LIVE_TASK[] !== nothing && !istaskdone(LIVE_TASK[])
    println("Live plot is already running. Run Cell 3 to stop it first.")
else
    GLMakie.activate!()

    sock = UDPSocket()
    bind(sock, listen_host, listen_port)
    LIVE_SOCK[] = sock
    LIVE_STOP[] = false

    xs = Float32[]
    ys = Float32[]
    zs = Float32[]
    rxs = Float64[]
    rys = Float64[]
    rzs = Float64[]
    cols = RGBf[]

    max_points = 4000
    refresh_every = 5
    max_packets = 20000

    norm01(v, lo, hi) = hi == lo ? 0.5 : (v - lo) / (hi - lo)

    # Build one GLMakie figure and update observables in-place.
    fig = Figure()
    ax = Axis3(
        fig[1, 1],
        xlabel = "X [m]",
        ylabel = "Y [m]",
        zlabel = "Z [m]",
        title = "Live TCP Position (XYZ), color from orientation (Rx,Ry,Rz)"
    )

    LIVE_POINTS[] = Point3f[]
    LIVE_COLORS[] = RGBf[]
    scatter!(ax, LIVE_POINTS; color = LIVE_COLORS, markersize = 5)

    screen = GLMakie.Screen(start_renderloop = true)
    display(screen, fig)
    LIVE_SCREEN[] = screen
    LIVE_FIG[] = fig

    LIVE_TASK[] = @async begin
        println("Listening on $(listen_host):$(listen_port) ...")
        k = 0
        recv_count = 0
        try
            while !LIVE_STOP[] && k < max_packets
                msg = String(recv(sock))
                if msg == "__STOP__"
                    continue
                end

                pkt = JSON3.read(msg)
                if !haskey(pkt, :actual_TCP_pose)
                    continue
                end
                pose = pkt.actual_TCP_pose

                x = Float32(pose[1])
                y = Float32(pose[2])
                z = Float32(pose[3])
                rx = Float64(pose[4])
                ry = Float64(pose[5])
                rz = Float64(pose[6])

                push!(xs, x); push!(ys, y); push!(zs, z)
                push!(rxs, rx); push!(rys, ry); push!(rzs, rz)

                r = norm01(rx, minimum(rxs), maximum(rxs))
                g = norm01(ry, minimum(rys), maximum(rys))
                b = norm01(rz, minimum(rzs), maximum(rzs))
                push!(cols, RGBf(r, g, b))

                if length(xs) > max_points
                    popfirst!(xs); popfirst!(ys); popfirst!(zs)
                    popfirst!(rxs); popfirst!(rys); popfirst!(rzs)
                    popfirst!(cols)
                end

                if k % refresh_every == 0
                    LIVE_POINTS[] = Point3f.(xs, ys, zs)
                    LIVE_COLORS[] = copy(cols)
                    yield()
                end

                recv_count += 1
                if recv_count == 1
                    println("First UDP packet received; live plot updating.")
                end

                k += 1
            end
        catch err
            println("Live plot task error: ", err)
            showerror(stdout, err, catch_backtrace())
            println()
        finally
            try
                close(sock)
            catch
            end
            LIVE_SOCK[] = nothing
            println("Live plot loop stopped.")
        end
    end

    errormonitor(LIVE_TASK[])
    println("Live plot started in background task. Run Cell 3 to stop.")
end

Live plot started in background task. Run Cell 3 to stop.
Listening on 127.0.0.1:9999 ...


In [12]:
using Sockets, JSON3

# UDP arrival test (run this when live plot cell is NOT running)
test_host = ip"127.0.0.1"
test_port = 9999
timeout_s = 5.0

sock = UDPSocket()

try
    bind(sock, test_host, test_port)
    println("Listening for UDP on $(test_host):$(test_port) for $(timeout_s) seconds...")

    msg_ref = Ref{Union{Nothing, String}}(nothing)
    err_ref = Ref{Union{Nothing, Any}}(nothing)

    recv_task = @async begin
        try
            msg_ref[] = String(recv(sock))
        catch err
            err_ref[] = err
        end
    end

    t0 = time()
    while !istaskdone(recv_task) && (time() - t0) < timeout_s
        sleep(0.05)
    end

    if !istaskdone(recv_task)
        close(sock)
        wait(recv_task)
        println("No UDP packet received within $(timeout_s) seconds.")
        println("Check that record.py is running and streaming to $(test_host):$(test_port).")
    elseif err_ref[] !== nothing
        println("UDP receive error: ", err_ref[])
    else
        msg = something(msg_ref[], "")
        println("UDP packet received.")
        println("Bytes: ", ncodeunits(msg))
        preview = first(msg, min(lastindex(msg), 180))
        println("Preview: ", preview)

        try
            pkt = JSON3.read(msg)
            if haskey(pkt, :actual_TCP_pose)
                pose = pkt.actual_TCP_pose
                println("actual_TCP_pose detected. XYZ = ", (pose[1], pose[2], pose[3]))
            else
                println("JSON parsed, but key :actual_TCP_pose was not found.")
            end
        catch
            println("Packet is not valid JSON (or not JSON3-compatible).")
        end
    end
catch err
    println("Could not bind UDP socket on $(test_host):$(test_port).")
    println("If Cell 2 is running, stop it first with Cell 4, then rerun this test.")
    println("Error: ", err)
finally
    try
        close(sock)
    catch
    end
end

Listening for UDP on 127.0.0.1:9999 for 5.0 seconds...
UDP packet received.
Bytes: 180
Preview: {"timestamp": 7381.295999999999, "actual_TCP_pose": [0.3394158900598096, -0.47418124691200114, 0.20844882878717022, -0.006652451563052258, 2.3724391895451453, 0.09534581477323596]}
actual_TCP_pose detected. XYZ = (0.3394158900598096, -0.47418124691200114, 0.20844882878717022)


In [11]:
using Sockets, GLMakie

if !isdefined(Main, :LIVE_STOP) || !isdefined(Main, :LIVE_TASK)
    println("Live controller not initialized. Run Cell 2 first.")
else
    LIVE_STOP[] = true

    # Send a local wake packet so recv(sock) unblocks immediately.
    if isdefined(Main, :LIVE_SOCK) && LIVE_SOCK[] !== nothing
        wake = UDPSocket()
        try
            host = isdefined(Main, :LIVE_HOST) ? LIVE_HOST : ip"127.0.0.1"
            port = isdefined(Main, :LIVE_PORT) ? LIVE_PORT : 9999
            send(wake, host, port, "__STOP__")
        finally
            close(wake)
        end
    end

    if LIVE_TASK[] !== nothing
        wait(LIVE_TASK[])
    end

    if isdefined(Main, :LIVE_SCREEN) && LIVE_SCREEN[] !== nothing
        try
            GLMakie.closeall()
        catch
        end
        LIVE_SCREEN[] = nothing
    end

    println("Stop request complete.")
end

Live plot loop stopped.
Stop request complete.
